# AmbiCode-Eval — Evaluation Demo

This notebook demonstrates the full evaluation workflow:
1. Running clean (baseline) prompt tests
2. Running ambiguous (perturbed) prompt tests
3. HumanEval-style execution
4. MBPP-style execution
5. DS-1000-style execution
6. End-to-end assessment: LLM call -> sandbox -> Ambiguity Tax

**Prerequisites:**
- `.env` file with `OPENROUTER_API_KEY`
- Docker Desktop running
- `pip install -e ".[dev]"`
- DS-1000 Docker image: `docker build -t ambicode-ds1000 -f docker/ds1000.Dockerfile .`

In [61]:
import json
import re
import sys
sys.path.insert(0, '..')

from src.data.model import BenchmarkItem, BenchmarkTask
from src.data.store import BenchmarkStore
from src.util.llm import LLMClient, ModelConfig
from src.util.sandbox import Sandbox


# ── Helpers ──────────────────────────────────────────────────────────────────

def strip_markdown_fences(code: str) -> str:
    """Strip markdown code fences from LLM output."""
    code = re.sub(r'^```(?:python)?\s*\n', '', code.strip())
    code = re.sub(r'\n```\s*$', '', code)
    return code.strip()


def extract_function_name(test_code: str) -> str:
    """Extract the expected function name from test assertions."""
    m = re.search(r'assert\s+(\w+)\s*\(', test_code)
    return m.group(1) if m else None


def build_prompt_with_name(prompt: str, test_code: str) -> str:
    """Add expected function name to prompt if detectable from tests."""
    func_name = extract_function_name(test_code)
    if func_name:
        return f"{prompt}\nThe function should be named `{func_name}`."
    return prompt


print('Helpers loaded OK')

Helpers loaded OK


In [62]:
# ── Load data and initialize clients ─────────────────────────────────────────

with open('../data/benchmark/benchmark.jsonl') as f:
    items = [BenchmarkItem.from_dict(json.loads(line)) for line in f if line.strip()]

store = BenchmarkStore.load_local('../data/raw')  # for HumanEval demo

client = LLMClient()
sandbox_default = Sandbox()                          # python:3.11-slim (MBPP, HumanEval)
sandbox_ds1000 = Sandbox(image='ambicode-ds1000')    # pandas, numpy, scipy, sklearn, etc.

system_prompt = (
    'You are a Python code generator. Write ONLY the Python function implementation. '
    'No explanation, no markdown fences, no extra text. Just the code.'
)

config = ModelConfig(model='gpt-5.4', temperature=0.0, max_tokens=1024)

print(f'Benchmark items: {len(items)}')
print(f'Raw tasks: {len(store)}')

Benchmark items: 62
Raw tasks: 1266


---
## 1. Clean Prompt Test (Baseline)

Send the original clean prompt to an LLM, then run its output against the original test suite.
This measures **baseline pass@k** — how well the model performs without ambiguity.

We use AMBI/002 (syntactic ambiguity) which has a clear clean prompt that models handle well.

In [63]:
# Pick AMBI/002 — syntactic ambiguity, reliable baseline pass
mbpp_item = next(it for it in items if it.task_id == 'AMBI/002')

print(f'Task: {mbpp_item.task_id} (anchor: {mbpp_item.anchor_task_id})')
print(f'Type: {mbpp_item.ambiguity_type}, Risk: {mbpp_item.risk_level}')
print(f'\n=== Clean Prompt ===')
print(mbpp_item.prompt)
print(f'\n=== Test Code ===')
print(mbpp_item.test_code)

Task: AMBI/002 (anchor: MBPP/240)
Type: syntactic, Risk: high

=== Clean Prompt ===
Write a function that takes in two lists and replaces the last element of the first list with the elements of the second list.

=== Test Code ===
assert replace_list([1, 3, 5, 7, 9, 10],[2, 4, 6, 8])==[1, 3, 5, 7, 9, 2, 4, 6, 8]
assert replace_list([1,2,3,4,5],[5,6,7,8])==[1,2,3,4,5,6,7,8]
assert replace_list(["red","blue","green"],["yellow"])==["red","blue","yellow"]


In [64]:
# Call LLM with clean prompt (include expected function name from tests)
clean_prompt = build_prompt_with_name(mbpp_item.prompt, mbpp_item.test_code)
print(f'Prompt sent to LLM:\n  {clean_prompt}\n')

clean_response = client.call(config, prompt=clean_prompt, system=system_prompt)
clean_code = strip_markdown_fences(clean_response.choices[0])

print(f'Model: {clean_response.model}')
print(f'Latency: {clean_response.latency_s}s')
print(f'\n=== Generated Code ===')
print(clean_code)

Prompt sent to LLM:
  Write a function that takes in two lists and replaces the last element of the first list with the elements of the second list.
The function should be named `replace_list`.

Model: openai/gpt-5.4
Latency: 1.826s

=== Generated Code ===
def replace_list(first_list, second_list):
    first_list[-1:] = second_list
    return first_list


In [65]:
# Run generated code against original test suite in sandbox
result = sandbox_default.run(clean_code, mbpp_item.test_code)

print(f'Passed: {result.passed}')
print(f'Exit code: {result.exit_code}')
if result.stderr:
    print(f'Stderr: {result.stderr[:300]}')

Passed: True
Exit code: 0


---
## 2. Ambiguous Prompt Test (Experimental)

Send the **perturbed prompt** (with injected ambiguity) to the same LLM.
Run the output against **both** test suites (dual-blind) to determine which interpretation the model chose.

In [66]:
print(f'=== Perturbed Prompt ===')
print(mbpp_item.perturbed_prompt)
print(f'\nInterpretation A: {mbpp_item.interpretation_a}')
print(f'Interpretation B: {mbpp_item.interpretation_b}')

=== Perturbed Prompt ===
def replace_list(list1, list2):
    """
    Write a function that takes in two lists and replaces the last element of the first list with the elements of the second list starting from the end.
    """

Interpretation A: The phrase 'starting from the end' modifies the replacement location in the first list, meaning the replacement targets the last element (matching the canonical slice assignment).
Interpretation B: The phrase 'starting from the end' modifies 'the elements of the second list', meaning the second list should be iterated in reverse order during the replacement.


In [67]:
# Call LLM with perturbed prompt
perturbed_prompt = build_prompt_with_name(mbpp_item.perturbed_prompt, mbpp_item.test_code)
perturbed_response = client.call(config, prompt=perturbed_prompt, system=system_prompt)
perturbed_code = strip_markdown_fences(perturbed_response.choices[0])

print(f'=== Generated Code (from ambiguous prompt) ===')
print(perturbed_code)

=== Generated Code (from ambiguous prompt) ===
def replace_list(list1, list2):
    return list1[:-1] + list2[::-1]


In [68]:
# Dual-blind execution: test against BOTH interpretations
result_a, result_b = sandbox_default.run_dual_blind(
    code=perturbed_code,
    test_a=mbpp_item.test_a,
    test_b=mbpp_item.test_b,
)

print(f'Test A (interpretation A): {"PASS" if result_a.passed else "FAIL"}')
print(f'Test B (interpretation B): {"PASS" if result_b.passed else "FAIL"}')
print()

if result_a.passed and not result_b.passed:
    print('Model chose Interpretation A (original meaning)')
elif result_b.passed and not result_a.passed:
    print('Model chose Interpretation B (alternative meaning)')
elif result_a.passed and result_b.passed:
    print('Model\'s code satisfies BOTH interpretations')
else:
    print('Model\'s code fails BOTH test suites')

Test A (interpretation A): FAIL
Test B (interpretation B): PASS

Model chose Interpretation B (alternative meaning)


---
## 3. HumanEval-Style Execution

HumanEval tasks provide a **function signature + docstring** as the prompt. The model writes the complete function. The test code contains a `check(candidate)` function that validates the output.

Note: HumanEval is not in the benchmark (0% Stage 4 pass rate), but we demo the execution format here.

In [69]:
# Load a HumanEval task from raw data
he_task = store.get('HumanEval/0')

print(f'Task: {he_task.task_id}')
print(f'Entry point: {he_task.entry_point}')
print(f'\n=== Prompt (function signature + docstring) ===')
print(he_task.prompt)
print(f'\n=== Canonical Solution (function body) ===')
print(he_task.canonical_solution)

Task: HumanEval/0
Entry point: has_close_elements

=== Prompt (function signature + docstring) ===
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


=== Canonical Solution (function body) ===
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False



In [70]:
# Ask LLM to write the complete function (not just the body)
# Asking for "just the body" leads to indentation issues, so we ask for the full function
he_response = client.call(config, prompt=he_task.prompt, system=system_prompt)
he_code = strip_markdown_fences(he_response.choices[0])

print(f'=== LLM Generated Code ===')
print(he_code)

=== LLM Generated Code ===
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    numbers = sorted(numbers)
    for i in range(1, len(numbers)):
        if numbers[i] - numbers[i - 1] < threshold:
            return True
    return False


In [71]:
# HumanEval execution: generated code + test code
# The LLM writes the complete function, test code calls check(function_name)
result = sandbox_default.run(he_code, he_task.test_code)

print(f'Passed: {result.passed}')
print(f'Exit code: {result.exit_code}')
if result.stderr:
    print(f'Error: {result.stderr.strip().split(chr(10))[-1][:200]}')

Passed: False
Exit code: 1
Error: NameError: name 'List' is not defined. Did you mean: 'list'?


---
## 4. MBPP-Style Execution

MBPP tasks have a natural-language prompt. The model writes a **complete, self-contained function**.
Test code is simple assert statements. Execution: `exec(code + test_code)`.

Here we verify the **exclusivity guarantee** — ref_a passes only test_a, ref_b passes only test_b.

In [72]:
print(f'Task: {mbpp_item.task_id} (anchor: {mbpp_item.anchor_task_id})')
print(f'\n=== Canonical Solution (ref_a) ===')
print(mbpp_item.canonical_solution)
print(f'\n=== Ref Solution B ===')
print(mbpp_item.ref_solution_b)
print(f'\n=== Test A ===')
print(mbpp_item.test_a)
print(f'\n=== Test B ===')
print(mbpp_item.test_b)

Task: AMBI/002 (anchor: MBPP/240)

=== Canonical Solution (ref_a) ===
def replace_list(list1,list2):
 list1[-1:] = list2
 replace_list=list1
 return replace_list


=== Ref Solution B ===
def replace_list(list1, list2):
    list1[-1:] = list2[::-1]
    return list1


=== Test A ===
assert replace_list([1, 3, 5, 7, 9, 10],[2, 4, 6, 8])==[1, 3, 5, 7, 9, 2, 4, 6, 8]
assert replace_list([1,2,3,4,5],[5,6,7,8])==[1,2,3,4,5,6,7,8]
assert replace_list(["red","blue","green"],["yellow"])==["red","blue","yellow"]

=== Test B ===
assert replace_list([1, 3, 5, 7, 9, 10], [2, 4, 6, 8]) == [1, 3, 5, 7, 9, 8, 6, 4, 2]
assert replace_list([1, 2, 3, 4, 5], [5, 6, 7, 8]) == [1, 2, 3, 4, 8, 7, 6, 5]
assert replace_list(["red", "blue", "green"], ["yellow", "orange"]) == ["red", "blue", "orange", "yellow"]



In [73]:
# Verify 2x2 exclusivity in sandbox
r_aa = sandbox_default.run(mbpp_item.canonical_solution, mbpp_item.test_a)
r_ab = sandbox_default.run(mbpp_item.canonical_solution, mbpp_item.test_b)
r_ba = sandbox_default.run(mbpp_item.ref_solution_b, mbpp_item.test_a)
r_bb = sandbox_default.run(mbpp_item.ref_solution_b, mbpp_item.test_b)

print(f'ref_a x test_a: {"PASS" if r_aa.passed else "FAIL"} (expected PASS)')
print(f'ref_a x test_b: {"PASS" if r_ab.passed else "FAIL"} (expected FAIL)')
print(f'ref_b x test_a: {"PASS" if r_ba.passed else "FAIL"} (expected FAIL)')
print(f'ref_b x test_b: {"PASS" if r_bb.passed else "FAIL"} (expected PASS)')
print(f'\nExclusivity: {"VERIFIED" if (r_aa.passed and not r_ab.passed and not r_ba.passed and r_bb.passed) else "FAILED"}')

ref_a x test_a: PASS (expected PASS)
ref_a x test_b: FAIL (expected FAIL)
ref_b x test_a: FAIL (expected FAIL)
ref_b x test_b: PASS (expected PASS)

Exclusivity: VERIFIED


---
## 5. DS-1000-Style Execution

DS-1000 tasks are data science problems (Pandas, NumPy, Scipy, Sklearn, etc.).

In the benchmark, DS-1000 items use a **normalized format**:
- `canonical_solution`: original code fragment wrapped as `__SOLUTION__ = r"""..."""`
- `test_code`: original DS-1000 harness + `test_execution(__SOLUTION__)`
- `ref_solution_b` and `test_b`: **self-contained** (include imports + test data)

Execution requires the `ambicode-ds1000` Docker image.

In [74]:
# Pick a DS-1000 benchmark item
ds_item = next(it for it in items if it.source == 'ds1000')

print(f'Task: {ds_item.task_id} (anchor: {ds_item.anchor_task_id})')
print(f'Library: {ds_item.library}')
print(f'Type: {ds_item.ambiguity_type}, Risk: {ds_item.risk_level}')
print(f'\n=== Prompt (first 400 chars) ===')
print(ds_item.prompt[:400])
print('...')

Task: AMBI/015 (anchor: DS1000/Pandas/227)
Library: Pandas
Type: coreferential, Risk: low

=== Prompt (first 400 chars) ===
Problem:
i need to create a dataframe containing tuples from a series of dataframes arrays. What I need is the following:
I have dataframes a and b:
a = pd.DataFrame(np.array([[1, 2],[3, 4]]), columns=['one', 'two'])
b = pd.DataFrame(np.array([[5, 6],[7, 8]]), columns=['one', 'two'])
c = pd.DataFrame(np.array([[9, 10],[11, 12]]), columns=['one', 'two'])
a:
   one  two
0    1    2
1    3    4
b: 
 
...


In [75]:
# DS-1000 canonical_solution is wrapped as __SOLUTION__
print('=== Canonical Solution (normalized, first 200 chars) ===')
print(ds_item.canonical_solution[:200])
print('...')

# Run canonical solution against original test harness
result = sandbox_ds1000.run(ds_item.canonical_solution, ds_item.test_code, timeout_s=60)
print(f'\nCanonical passes test_a: {result.passed}')
if not result.passed and result.stderr:
    print(f'Error: {result.stderr.strip().split(chr(10))[-1][:200]}')

=== Canonical Solution (normalized, first 200 chars) ===
__SOLUTION__ = r"""def g(a,b,c):
    return pd.DataFrame(np.rec.fromarrays((a.values, b.values, c.values)).tolist(),columns=a.columns,index=a.index)

result = g(a.copy(),b.copy(), c.copy())
"""

...

Canonical passes test_a: True


In [76]:
# DS-1000 ref_solution_b and test_b are self-contained
print('=== Ref Solution B (self-contained, first 300 chars) ===')
print(ds_item.ref_solution_b[:300])
print('...')

# Run ref_solution_b against test_b (both self-contained)
result_b = sandbox_ds1000.run(ds_item.ref_solution_b, ds_item.test_b, timeout_s=60)
print(f'\nRef solution B passes test_b: {result_b.passed}')
if not result_b.passed and result_b.stderr:
    print(f'Error: {result_b.stderr.strip().split(chr(10))[-1][:200]}')

=== Ref Solution B (self-contained, first 300 chars) ===
import pandas as pd
import numpy as np

a = pd.DataFrame(np.array([[1, 2],[3, 4]]), columns=['one', 'two'])
b = pd.DataFrame(np.array([[5, 6],[7, 8]]), columns=['one', 'two'])
c = pd.DataFrame(np.array([[9, 10],[11, 12]]), columns=['one', 'two'])

def g(a, b, c):
    return pd.DataFrame(np.rec.froma
...

Ref solution B passes test_b: True


---
## 6. End-to-End Assessment

Full workflow: call LLM with both prompts -> sandbox execution -> compute Ambiguity Tax.

We call the model 3 times per prompt (simulating n=3 sampling) and compute pass rates.

In [77]:
eval_item = mbpp_item  # reuse AMBI/002

print(f'Evaluating: {eval_item.task_id} ({eval_item.ambiguity_type})')
print(f'\nClean prompt:\n  {eval_item.prompt}')
print(f'\nPerturbed prompt:\n  {eval_item.perturbed_prompt[:200]}...')
print(f'\nInterpretation A: {eval_item.interpretation_a}')
print(f'Interpretation B: {eval_item.interpretation_b}')

Evaluating: AMBI/002 (syntactic)

Clean prompt:
  Write a function that takes in two lists and replaces the last element of the first list with the elements of the second list.

Perturbed prompt:
  def replace_list(list1, list2):
    """
    Write a function that takes in two lists and replaces the last element of the first list with the elements of the second list starting from the end.
    """...

Interpretation A: The phrase 'starting from the end' modifies the replacement location in the first list, meaning the replacement targets the last element (matching the canonical slice assignment).
Interpretation B: The phrase 'starting from the end' modifies 'the elements of the second list', meaning the second list should be iterated in reverse order during the replacement.


In [78]:
# Sample multiple responses by calling the model 3 times with temperature > 0
# (OpenRouter may not support n>1 for all models, so we call multiple times)
sample_config = ModelConfig(model='gpt-5.4', temperature=0.8, max_tokens=1024)
n_samples = 3

# ── Baseline (clean prompt) ──
baseline_prompt = build_prompt_with_name(eval_item.prompt, eval_item.test_code)
baseline_pass = 0

print('=== Baseline (clean prompt) ===')
for i in range(n_samples):
    resp = client.call(sample_config, prompt=baseline_prompt, system=system_prompt)
    code = strip_markdown_fences(resp.choices[0])
    r = sandbox_default.run(code, eval_item.test_code)
    baseline_pass += int(r.passed)
    print(f'  Sample {i+1}: {"PASS" if r.passed else "FAIL"}')

baseline_rate = baseline_pass / n_samples
print(f'  Baseline pass rate: {baseline_rate:.1%}')

=== Baseline (clean prompt) ===
  Sample 1: PASS
  Sample 2: PASS
  Sample 3: PASS
  Baseline pass rate: 100.0%


In [79]:
# ── Perturbed (ambiguous prompt) ──
perturbed_prompt = build_prompt_with_name(eval_item.perturbed_prompt, eval_item.test_code)
perturbed_results = []

print('=== Perturbed (ambiguous prompt) ===')
for i in range(n_samples):
    resp = client.call(sample_config, prompt=perturbed_prompt, system=system_prompt)
    code = strip_markdown_fences(resp.choices[0])
    
    r_a, r_b = sandbox_default.run_dual_blind(
        code=code, test_a=eval_item.test_a, test_b=eval_item.test_b,
    )
    
    perturbed_results.append({'pass_a': r_a.passed, 'pass_b': r_b.passed})
    
    interp = 'A' if r_a.passed and not r_b.passed else \
             'B' if r_b.passed and not r_a.passed else \
             'Both' if r_a.passed and r_b.passed else 'Neither'
    
    print(f'  Sample {i+1}: test_a={"P" if r_a.passed else "F"}  '
          f'test_b={"P" if r_b.passed else "F"}  -> {interp}')

=== Perturbed (ambiguous prompt) ===
  Sample 1: test_a=F  test_b=P  -> B
  Sample 2: test_a=P  test_b=F  -> A
  Sample 3: test_a=F  test_b=P  -> B


In [80]:
# ── Compute Ambiguity Tax ──
pass_a_rate = sum(r['pass_a'] for r in perturbed_results) / n_samples
pass_b_rate = sum(r['pass_b'] for r in perturbed_results) / n_samples
pass_either_rate = sum(r['pass_a'] or r['pass_b'] for r in perturbed_results) / n_samples

chose_a = sum(r['pass_a'] and not r['pass_b'] for r in perturbed_results)
chose_b = sum(r['pass_b'] and not r['pass_a'] for r in perturbed_results)

ambiguity_tax = baseline_rate - pass_either_rate

print(f'=== Results ===')
print(f'  Baseline pass rate:   {baseline_rate:.1%}')
print(f'  Perturbed pass@1(A):  {pass_a_rate:.1%}')
print(f'  Perturbed pass@1(B):  {pass_b_rate:.1%}')
print(f'  Perturbed pass@1(any):{pass_either_rate:.1%}')
print(f'  Chose A: {chose_a}/{n_samples}  Chose B: {chose_b}/{n_samples}')
print(f'\n=== Ambiguity Tax ===')
print(f'  {ambiguity_tax:+.1%}')
if ambiguity_tax > 0:
    print(f'  -> Model loses {ambiguity_tax:.0%} performance due to ambiguity.')
elif ambiguity_tax == 0:
    print(f'  -> No ambiguity tax detected.')
else:
    print(f'  -> Negative tax (model does better on ambiguous prompt).')

=== Results ===
  Baseline pass rate:   100.0%
  Perturbed pass@1(A):  33.3%
  Perturbed pass@1(B):  66.7%
  Perturbed pass@1(any):100.0%
  Chose A: 1/3  Chose B: 2/3

=== Ambiguity Tax ===
  +0.0%
  -> No ambiguity tax detected.


---
## Summary

| Step | What | API |
|------|------|-----|
| Clean prompt test | Baseline pass@k | `client.call()` + `sandbox.run()` |
| Ambiguous prompt test | Perturbed pass@k | `client.call()` + `sandbox.run_dual_blind()` |
| HumanEval execution | prompt + body + check() | `sandbox.run(prompt + code, test_code)` |
| MBPP execution | function + asserts | `sandbox.run(code, test_code)` |
| DS-1000 execution | \_\_SOLUTION\_\_ + harness | `sandbox.run(code, test_code)` with `ambicode-ds1000` image |
| Assessment | LLM -> sandbox -> Ambiguity Tax | Multiple calls + dual-blind |

For the full-scale evaluation across all 62 items and multiple models, see `scripts/`.